# 02. Price-only event definition and eligibility audit

Here I ask why the Bitcoin early-warning rules I tested earlier did not catch crashes. I define crashes from prices alone and audit, event by event, whether each crash was catchable at all; I do not optimize thresholds.

- I read the one-minute Binance spot BTCUSDT cache I build in notebook 01 (`BTCUSDT_1m_regularized.parquet` and `btc_signals_1m.parquet` in `MyDrive/btc_lead_cost_results_77months_spot/cache/`), 2020-01 to 2026-04, and evaluate events from 2023-01 to 2026-04.
- I define a crash as a close 4% below the highest close of the previous 240 minutes, confirmed by five consecutive closes, with a 24-hour refractory period. I run the fixed and cost-optimized spectral and volatility rules unchanged as diagnostics, take the optimized cut-offs from notebook 01's `selected_rules_2021_2022.json`, and give each missed event a reason.
- This audit is a retrospective diagnostic. I had already explored the data from 2023 on, so the results are not out-of-sample evidence.
- I write the results to `event_definition_eligibility_audit/`, including `price_only_event_catalog.csv`, which later notebooks read.


## Event definition and eligibility layers

I predeclared the primary outcome for the audit.
- From prices alone, a crash begins when a one-minute close falls 4% below the highest close of the preceding 240 minutes.
- I require five consecutive observed closes at or below the level set at that first breach.
- I anchor the event at the first breach (`breach_utc`); `confirmed_utc` is four minutes later. The reference peak uses past closes only. For a stricter pre-decline diagnostic I also record the first 1% drop from the same peak.
- Later breaches belong to the same episode unless a 24-hour refractory period has passed and a new local peak has formed after the previous confirmed event.
- A true early warning comes 30–240 minutes before the first breach; alerts after the 4% breach do not count.

I use two eligibility layers, neither of which is part of the crash definition:
1. `common_valid`: an observed price bar and finite values for all model input scores.
2. `lowvol_gate`: `common_valid` plus the low-volatility filter `price_lowvol==1`.

I run the fixed and cost-optimized spectral and volatility threshold rules unchanged as diagnostics, not as a test of a newly fitted model. Because the cache contains short gap-fill bars, I exclude minutes with `volume==0`.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Colab: use local Parquet copies if available.')

BASE = Path('/content/drive/MyDrive/btc_lead_cost_results_77months_spot')
# --- SPOT guard: this notebook may only read the spot cache built and fingerprint-verified by notebook 1 ---
import json as _json
_src_file = BASE/'cache'/'DATA_SOURCE.json'
if not _src_file.is_file() or _json.loads(_src_file.read_text()).get('venue') != 'spot':
    raise RuntimeError(f'Not a spot cache: {_src_file} is missing or venue!=spot. Run notebook 1 (1_..._SPOT) to the end first.')
print('SPOT cache verified:', _json.loads(_src_file.read_text())['market'])

CACHE = BASE/'cache'
OUT = BASE/'event_definition_eligibility_audit'
OUT.mkdir(parents=True, exist_ok=True)
PRICE = CACHE/'BTCUSDT_1m_regularized.parquet'
SIGNAL = CACHE/'btc_signals_1m.parquet'
assert PRICE.exists() and SIGNAL.exists(), f'Missing caches: {PRICE}, {SIGNAL}'

btc = pd.read_parquet(PRICE, columns=['close','volume']).sort_index()
sig = pd.read_parquet(SIGNAL).sort_index()
assert isinstance(btc.index,pd.DatetimeIndex) and isinstance(sig.index,pd.DatetimeIndex)
assert btc.index.tz is not None and sig.index.tz is not None, 'Expected tz-aware UTC index'
btc.index=btc.index.tz_convert('UTC'); sig.index=sig.index.tz_convert('UTC')
assert btc.index.equals(sig.index), 'Cache indexes differ; both files need the same market and minute grid.'
assert btc.index.is_unique and btc.index.is_monotonic_increasing
assert np.all(np.diff(btc.index.asi8)==60*10**9), 'Expected complete 1m UTC index after regularization'
for c in ['z_hs','z_vol','tsr','price_lowvol','event']:
    assert c in sig.columns, f'Missing signal field {c}'
OBS = np.isfinite(btc.close.to_numpy(dtype=float)) & (btc.close.to_numpy(dtype=float)>0) & (btc.volume.to_numpy(dtype=float)>0)
print('UTC period:', btc.index[0], 'to', btc.index[-1])
print('Minutes:',len(btc), 'observed positive-volume bars:',int(OBS.sum()), 'excluded zero/missing bars:',int((~OBS).sum()))
print('Source: Binance SPOT (fingerprint-verified by notebook 1).')

Mounted at /content/drive
SPOT cache verified: Binance spot BTCUSDT 1m klines
UTC period: 2020-01-01 00:00:00+00:00 to 2026-04-30 23:59:00+00:00
Minutes: 3329280 observed positive-volume bars: 3326744 excluded zero/missing bars: 2536
Source: Binance SPOT (fingerprint-verified by notebook 1).


In [2]:
H=240                      # past-price peak lookback, minutes
DECLINE=0.04               # 4% from past rolling peak
SUSTAIN=5                  # five consecutive minute closes below the same fixed level
REFRACTORY_MIN=1440        # 24 hours before counting a different crash
MIN_LEAD=30                # minimum minutes before 4% breach to call it actionable warning
MAX_LEAD=240               # maximum early warning lookback
assert 0<MIN_LEAD<MAX_LEAD


def detect_price_only_crashes(close, observed, index, H=240, decline=.04,
                              sustain=5, refractory=1440):
    """Ex-post *price-only* event catalog; no signal, low-volatility, or future label in definition.
       A 5-close confirmation is only used to decide an historical event is real.
       `breach_utc` (not `confirmed_utc`) is the strict warning deadline.
    """
    c=np.asarray(close,dtype=np.float64)
    observed=np.asarray(observed,dtype=bool)
    assert len(c)==len(observed)==len(index)
    history=pd.Series(np.where(observed,c,np.nan))
    prior_peak=history.rolling(H,min_periods=H).max().shift(1).to_numpy()
    raw=(observed & np.isfinite(prior_peak) & (c<=(1-decline)*prior_peak))
    fresh=raw & ~np.r_[False,raw[:-1]]
    rows=[]; last_confirm=-10**12
    for i in np.flatnonzero(fresh):
        if i+sustain>len(c): continue
        if i-last_confirm<refractory: continue
        cut=(1-decline)*prior_peak[i]
        if not (observed[i:i+sustain].all() and np.all(c[i:i+sustain]<=cut)):
            continue
        prev=c[i-H:i]
        if len(prev)!=H or not observed[i-H:i].all():continue
        peak_i=int(i-H+np.argmax(prev))
        if peak_i <= last_confirm:continue  # require a new local peak after prior event
        onset_i=peak_i+1+int(np.flatnonzero(c[peak_i+1:i+1]<=0.99*c[peak_i])[0])
        rows.append({'event_id':len(rows)+1,
                     'peak_utc':index[peak_i], 'onset_1pct_utc':index[onset_i], 'breach_utc':index[i],
                     'confirmed_utc':index[i+sustain-1],
                     'peak_close':float(c[peak_i]),
                     'breach_close':float(c[i]),
                     'decline_at_breach_pct':float((c[i]/c[peak_i]-1)*100),
                     'peak_to_breach_minutes':int(i-peak_i),
                     'onset_to_breach_minutes':int(i-onset_i)})
        last_confirm=i+sustain-1
    return pd.DataFrame(rows)

catalog=detect_price_only_crashes(btc.close.to_numpy(),OBS,btc.index,
          H=H,decline=DECLINE,sustain=SUSTAIN,refractory=REFRACTORY_MIN)
assert len(catalog)>0, 'No price-only events found; inspect data and rule.'
assert catalog['breach_utc'].is_monotonic_increasing
catalog['year']=catalog.breach_utc.dt.year
catalog.to_csv(OUT/'price_only_event_catalog.csv',index=False)
print('All price-only events:',len(catalog))
display(catalog.groupby('year',as_index=False).agg(events=('event_id','count'),
            median_peak_to_breach_min=('peak_to_breach_minutes','median')))
display(catalog.tail(12))

All price-only events: 269


,year,events,median_peak_to_breach_min
0,2020,42,170.0
1,2021,89,198.0
2,2022,55,194.0
3,2023,17,180.0
4,2024,36,173.5
5,2025,21,184.0
6,2026,9,176.0


,event_id,peak_utc,onset_1pct_utc,breach_utc,confirmed_utc,peak_close,breach_close,decline_at_breach_pct,peak_to_breach_minutes,onset_to_breach_minutes,year
257,258,2025-12-15 14:37:00+00:00,2025-12-15 14:46:00+00:00,2025-12-15 16:46:00+00:00,2025-12-15 16:50:00+00:00,89876.41,86205.07,-4.084876,129,120,2025
258,259,2025-12-17 15:06:00+00:00,2025-12-17 15:41:00+00:00,2025-12-17 16:21:00+00:00,2025-12-17 16:25:00+00:00,90297.32,86547.99,-4.152205,75,40,2025
259,260,2025-12-18 15:39:00+00:00,2025-12-18 16:01:00+00:00,2025-12-18 19:01:00+00:00,2025-12-18 19:05:00+00:00,89318.98,85669.15,-4.086287,202,180,2025
260,261,2026-01-29 13:29:00+00:00,2026-01-29 14:57:00+00:00,2026-01-29 16:07:00+00:00,2026-01-29 16:11:00+00:00,88167.98,84488.06,-4.173760,158,70,2026
261,262,2026-01-31 14:06:00+00:00,2026-01-31 14:23:00+00:00,2026-01-31 17:08:00+00:00,2026-01-31 17:12:00+00:00,82928.16,79474.71,-4.164388,182,165,2026
262,263,2026-02-02 00:38:00+00:00,2026-02-02 01:09:00+00:00,2026-02-02 03:34:00+00:00,2026-02-02 03:38:00+00:00,78106.99,74870.65,-4.143470,176,145,2026
263,264,2026-02-03 14:29:00+00:00,2026-02-03 14:54:00+00:00,2026-02-03 17:42:00+00:00,2026-02-03 17:46:00+00:00,78425.83,75263.44,-4.032332,193,168,2026
264,265,2026-02-04 14:36:00+00:00,2026-02-04 14:56:00+00:00,2026-02-04 17:50:00+00:00,2026-02-04 17:54:00+00:00,75500.04,72391.33,-4.117495,194,174,2026
265,266,2026-02-05 16:55:00+00:00,2026-02-05 17:04:00+00:00,2026-02-05 19:46:00+00:00,2026-02-05 19:50:00+00:00,68585.84,65709.45,-4.193854,171,162,2026
266,267,2026-02-07 04:22:00+00:00,2026-02-07 04:37:00+00:00,2026-02-07 07:04:00+00:00,2026-02-07 07:08:00+00:00,71666.62,68776.36,-4.032924,162,147,2026


In [3]:
# Unit test before interpreting real counts. A 100 -> 95 drop held for 5m must create one event.
idx=pd.date_range('2021-01-01', periods=2200, freq='min',tz='UTC')
test=np.r_[np.full(1000,100.),np.full(1200,95.)]
toy=detect_price_only_crashes(test,np.ones(len(test),dtype=bool),idx)
assert len(toy)==1, f'Expected exactly one event; got {len(toy)}'
assert toy.iloc[0].breach_utc==idx[1000]
assert toy.iloc[0].confirmed_utc==idx[1004]
assert toy.iloc[0].peak_utc < toy.iloc[0].breach_utc
assert toy.iloc[0].onset_1pct_utc == toy.iloc[0].breach_utc
# A >4% one-minute drop that immediately rebounds is not a 5-minute sustained event.
rebound=np.r_[np.full(1000,100.),95.,np.full(1199,100.)]
assert len(detect_price_only_crashes(rebound,np.ones(len(rebound),dtype=bool),idx))==0
print('Synthetic event-definition tests passed.')

Synthetic event-definition tests passed.


In [4]:
# Eligible score minutes are not event labels. First audit input/data availability, then the low-vol gate.
z_hs=sig.z_hs.to_numpy(dtype=np.float64)
z_vol=sig.z_vol.to_numpy(dtype=np.float64)
ts=sig.tsr.to_numpy(dtype=np.float64)
common_valid=OBS & np.isfinite(z_hs) & np.isfinite(z_vol) & np.isfinite(ts) & (ts>=0) & (ts<=1)
lowvol_gate=common_valid & (sig.price_lowvol.to_numpy(dtype=np.float64)==1.0)

# 'optimized' cut-offs come from notebook 1's 2021–2022 selection on this spot cache.
_sel_path=BASE/'selected_rules_2021_2022.json'
if not _sel_path.is_file():
    raise FileNotFoundError(f'Run notebook 1 to the end to create {_sel_path}.')
_sel=_json.loads(_sel_path.read_text())
POLICIES={
    'fixed_H+F': ('z_hs',1.0,0.60),
    'optimized_H+F': ('z_hs',float(_sel['z_hs'][0]),float(_sel['z_hs'][1])),
    'optimized_V+F': ('z_vol',float(_sel['z_vol'][0]),float(_sel['z_vol'][1])),
}
print('Rules from spot 2021–2022 selection:',POLICIES)

# This scanner applies the 240-minute cooldown, without a per-month cap.
def scan_alerts(times_ns, raw_trigger, cooldown=H):
    raw_idxs=np.flatnonzero(np.asarray(raw_trigger,dtype=bool))
    selected=[]; last=-10**15
    for k in raw_idxs:
        t=int(times_ns[k])
        if t-last>=cooldown*60*10**9:
            selected.append(t)
            last=t
    return np.asarray(selected,dtype=np.int64)

scores={'z_hs':z_hs,'z_vol':z_vol}
trigger_masks={}
alerts_by_policy={}
for name,(score,zcut,tscut) in POLICIES.items():
    raw=lowvol_gate & (scores[score]>zcut) & (ts>tscut)
    trigger_masks[name]=raw
    alerts_by_policy[name]=scan_alerts(btc.index.asi8,raw)
    print(f'{name:20s}: eligible raw minutes={raw.sum():,}; scheduled alerts={len(alerts_by_policy[name]):,}')

# Audit eligibility before estimating any predictive superiority.
def in_warning_window(event_time_ns, times_ns, min_lead=MIN_LEAD, max_lead=MAX_LEAD):
    lower=event_time_ns-max_lead*60*10**9
    upper=event_time_ns-min_lead*60*10**9
    lo=int(np.searchsorted(times_ns,lower,'left'))
    hi=int(np.searchsorted(times_ns,upper,'right'))
    return lo,hi

EV_START=pd.Timestamp('2023-01-01',tz='UTC')
EV_END=min(pd.Timestamp('2026-05-01',tz='UTC'), btc.index[-1]+pd.Timedelta(minutes=1))
chosen=catalog[(catalog.breach_utc>=EV_START+pd.Timedelta(minutes=MAX_LEAD)) &
               (catalog.breach_utc<EV_END)].copy().reset_index(drop=True)
assert len(chosen)>0,'No evaluation episodes: check exact criteria and the time range.'
rows=[]
for e in chosen.itertuples(index=False):
    lo,hi=in_warning_window(e.breach_utc.value,btc.index.asi8)
    row={'event_id':int(e.event_id), 'breach_utc':e.breach_utc,
         'onset_1pct_utc':e.onset_1pct_utc,
         'confirmed_utc':e.confirmed_utc, 'year':int(e.year),
         'window_total_minutes':max(0,hi-lo),
         'common_valid_minutes':int(common_valid[lo:hi].sum()),
         'lowvol_gate_minutes':int(lowvol_gate[lo:hi].sum())}
    for name in POLICIES:
        row[f'{name}__raw_trigger_minutes']=int(trigger_masks[name][lo:hi].sum())
    rows.append(row)
eligibility=pd.DataFrame(rows)
assert (eligibility['lowvol_gate_minutes']<=eligibility['common_valid_minutes']).all()
eligibility.to_csv(OUT/'event_eligibility_audit.csv',index=False)
display(eligibility[['event_id','breach_utc','common_valid_minutes','lowvol_gate_minutes']+
                    [f'{p}__raw_trigger_minutes' for p in POLICIES]].head(30))
print('Events with no valid candidate minutes:',int((eligibility.common_valid_minutes==0).sum()))
print('Events excluded entirely by the low-volatility gate:',
      int(((eligibility.common_valid_minutes>0)&(eligibility.lowvol_gate_minutes==0)).sum()))

Rules from spot 2021–2022 selection: {'fixed_H+F': ('z_hs', 1.0, 0.6), 'optimized_H+F': ('z_hs', 2.0, 0.55), 'optimized_V+F': ('z_vol', 1.75, 0.5)}
fixed_H+F           : eligible raw minutes=20,134; scheduled alerts=2,092
optimized_H+F       : eligible raw minutes=4,219; scheduled alerts=785
optimized_V+F       : eligible raw minutes=5,636; scheduled alerts=424


,event_id,breach_utc,common_valid_minutes,lowvol_gate_minutes,fixed_H+F__raw_trigger_minutes,optimized_H+F__raw_trigger_minutes,optimized_V+F__raw_trigger_minutes
0,187,2023-01-18 16:21:00+00:00,211,0,0,0,0
1,188,2023-02-16 23:24:00+00:00,211,0,0,0,0
2,189,2023-03-03 01:31:00+00:00,211,43,0,0,0
3,190,2023-03-09 20:11:00+00:00,211,0,0,0,0
4,191,2023-03-14 19:12:00+00:00,211,0,0,0,0
5,192,2023-03-22 19:05:00+00:00,211,0,0,0,0
6,193,2023-03-27 15:02:00+00:00,211,0,0,0,0
7,194,2023-04-26 19:38:00+00:00,211,0,0,0,0
8,195,2023-05-10 17:20:00+00:00,211,0,0,0,0
9,196,2023-06-05 17:09:00+00:00,211,0,0,0,0


Events with no valid candidate minutes: 0
Events excluded entirely by the low-volatility gate: 76


In [5]:
# Match each emitted alert at most once to a price-only episode (strictly before the first 4% breach).
# This computes diagnostic outcomes only; the notebook 01 thresholds were chosen under a different label.

def match_alarms(events, alerts_ns, tmin=MIN_LEAD, tmax=MAX_LEAD):
    a=np.asarray(alerts_ns,dtype=np.int64)
    used=set(); matching={}
    for e in events.itertuples(index=False):
        stop=int(e.breach_utc.value)
        lo=int(np.searchsorted(a,stop-tmax*60*10**9,'left'))
        hi=int(np.searchsorted(a,stop-tmin*60*10**9,'right'))
        poss=[j for j in range(lo,hi) if j not in used]
        if poss:
            j=poss[0]  # earliest alert gives longer time to respond
            used.add(j)
            matching[int(e.event_id)]= {'alarm_utc':pd.Timestamp(int(a[j]),tz='UTC'),
                                       'lead_to_breach_min':float((stop-int(a[j]))/(60*10**9))}
    return matching

full_audit=eligibility.copy()
summary=[]
for policy,alarms in alerts_by_policy.items():
    scored_alerts=alarms[(alarms>=EV_START.value)&
        (alarms<EV_END.value-MAX_LEAD*60*10**9)]
    matched=match_alarms(chosen,scored_alerts)
    reasons=[]
    for _,e in full_audit.iterrows():
        if int(e.event_id) in matched:
            reason='caught'
        elif e.common_valid_minutes==0:
            reason='no_complete_feature_minutes'
        elif e.lowvol_gate_minutes==0:
            reason='blocked_by_lowvol_gate'
        elif e[policy+'__raw_trigger_minutes']==0:
            reason='failed_score_or_flow_threshold'
        else:
            reason='trigger_exists_but_no_emitted_alert_in_window'
        reasons.append(reason)
    full_audit[f'{policy}__outcome']=reasons
    full_audit[f'{policy}__alarm_utc']=[matched.get(int(i),{}).get('alarm_utc',pd.NaT)
                                           for i in full_audit.event_id]
    full_audit[f'{policy}__lead_to_breach_min']=[matched.get(int(i),{}).get('lead_to_breach_min',np.nan)
                                           for i in full_audit.event_id]
    full_audit[f'{policy}__before_1pct_onset']=[bool(int(e.event_id) in matched and
         matched[int(e.event_id)]['alarm_utc']<e.onset_1pct_utc)
         for _,e in full_audit.iterrows()]
    n=len(scored_alerts); caught=len(matched)
    summary.append({'policy':policy,'evaluation_start':EV_START,
                    'evaluation_end_exclusive':EV_END,
                    'events':len(chosen),'alarms_scored':n,'events_caught':caught,
                    'event_recall':caught/len(chosen),
                    'caught_before_1pct_onset':int(full_audit[f'{policy}__before_1pct_onset'].sum()),
                    'false_alarms_scored':n-caught,
                    'mean_lead_to_first_breach_min':np.mean([x['lead_to_breach_min'] for x in matched.values()]) if matched else np.nan})

full_audit.to_csv(OUT/'event_by_event_missed_reason.csv',index=False)
comparison=pd.DataFrame(summary)
comparison.to_csv(OUT/'price_only_policy_diagnostic.csv',index=False)
reason_cols=[c for c in full_audit.columns if c.endswith('__outcome')]
reason_breakdown=pd.concat([full_audit[c].value_counts().rename(c.split('__')[0]) for c in reason_cols],axis=1).fillna(0).astype(int)
reason_breakdown.to_csv(OUT/'missed_reason_breakdown.csv')
print('Price-only diagnostic (retrospective, not a confirmatory estimate)')
display(comparison)
print('ROOT CAUSES OF MISSED EVENTS')
display(reason_breakdown)

Price-only diagnostic (retrospective, not a confirmatory estimate)


,policy,evaluation_start,evaluation_end_exclusive,events,alarms_scored,events_caught,event_recall,caught_before_1pct_onset,false_alarms_scored,mean_lead_to_first_breach_min
0,fixed_H+F,2023-01-01 00:00:00+00:00,2026-05-01 00:00:00+00:00,83,1026,4,0.048193,4,1022,104.750000
1,optimized_H+F,2023-01-01 00:00:00+00:00,2026-05-01 00:00:00+00:00,83,332,3,0.036145,0,329,53.333333
2,optimized_V+F,2023-01-01 00:00:00+00:00,2026-05-01 00:00:00+00:00,83,168,1,0.012048,0,167,39.000000


ROOT CAUSES OF MISSED EVENTS


,fixed_H+F,optimized_H+F,optimized_V+F
blocked_by_lowvol_gate,76,76,76
caught,4,3,1
trigger_exists_but_no_emitted_alert_in_window,2,0,0
failed_score_or_flow_threshold,1,4,6


In [6]:
# Diagnostic sensitivity: vary the price-only event definition without replacing the primary results.
# Any definition chosen after looking at outcomes is exploratory, not confirmatory.
sens=[]
for decline in (0.03,0.04,0.05):
    for refrac in (720,1440,2880):
        e=detect_price_only_crashes(btc.close.to_numpy(),OBS,btc.index,
                            H=H,decline=decline,sustain=SUSTAIN,refractory=refrac)
        e=e[(e.breach_utc>=EV_START)&(e.breach_utc<EV_END)]
        sens.append({'decline_pct':decline*100,'refractory_hours':refrac/60,
                     'eval_event_count':len(e)})
sensitivity=pd.DataFrame(sens)
sensitivity.to_csv(OUT/'definition_sensitivity_counts.csv',index=False)
print('Sensitivity in historical event counts (no model selection on these numbers)')
display(sensitivity)

Sensitivity in historical event counts (no model selection on these numbers)


,decline_pct,refractory_hours,eval_event_count
0,3.0,12.0,210
1,3.0,24.0,183
2,3.0,48.0,148
3,4.0,12.0,91
4,4.0,24.0,83
5,4.0,48.0,69
6,5.0,12.0,45
7,5.0,24.0,42
8,5.0,48.0,38


## Interpretation
- In `event_eligibility_audit.csv`, an event with `common_valid_minutes==0` has no usable feature minutes in its warning window, and the low-volatility regime filter excludes an event with `lowvol_gate_minutes==0` entirely, so no rule with that gate can catch it.
- `event_by_event_missed_reason.csv` separates failures of the score from scheduler and cooldown effects.
- In `price_only_policy_diagnostic.csv` I re-evaluate the notebook 01 thresholds on the price-only events; it is not a comparison of retrained models.
- `caught_before_1pct_onset` counts matched alerts issued before the first 1% fall from the event's preceding peak. The primary target remains the 4% breach with a 30–240-minute lead, and I report both.